In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "labelled_training_data.csv")
val_df   = pd.read_csv(DATA_DIR / "labelled_validation_data.csv")
test_df  = pd.read_csv(DATA_DIR / "labelled_testing_data.csv")

print(f"Loaded: Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")
print(f"Columns: {train_df.columns.tolist()}")

Loaded: Train=763,144 | Val=188,967 | Test=188,967
Columns: ['timestamp', 'processId', 'threadId', 'parentProcessId', 'userId', 'mountNamespace', 'processName', 'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum', 'returnValue', 'args', 'sus', 'evil']


In [2]:
def bin_uid(df):
    df = df.copy()
    df['uid_root']    = (df['userId'] == 0).astype(int)
    df['uid_daemon']  = ((df['userId'] >= 1) & (df['userId'] <= 999)).astype(int)
    df['uid_external']= (df['userId'] >= 1000).astype(int)
    return df

train_df = bin_uid(train_df)
val_df   = bin_uid(val_df)
test_df  = bin_uid(test_df)

print("=== UID BINS IN TEST SET ===")
print(f"uid_root     : {test_df['uid_root'].sum():,}")
print(f"uid_daemon   : {test_df['uid_daemon'].sum():,}")
print(f"uid_external : {test_df['uid_external'].sum():,}")
print("\nUID binning done ✅")

=== UID BINS IN TEST SET ===
uid_root     : 27,495
uid_daemon   : 221
uid_external : 161,251

UID binning done ✅


In [3]:
def add_return_features(df):
    df = df.copy()
    df['is_failure']    = (df['returnValue'] != 0).astype(int)
    df['is_success']    = (df['returnValue'] == 0).astype(int)
    return df

train_df = add_return_features(train_df)
val_df   = add_return_features(val_df)
test_df  = add_return_features(test_df)

print("=== FAILURE RATE ===")
print(f"Training failure rate : {train_df['is_failure'].mean()*100:.2f}%")
print(f"Test failure rate     : {test_df['is_failure'].mean()*100:.2f}%")

# Compare failure rate in evil vs benign events
evil_fail    = test_df[test_df['evil']==1]['is_failure'].mean()*100
benign_fail  = test_df[test_df['evil']==0]['is_failure'].mean()*100
print(f"\nEvil events failure rate   : {evil_fail:.2f}%")
print(f"Benign events failure rate : {benign_fail:.2f}%")
print("\nReturn value features done ✅")


=== FAILURE RATE ===
Training failure rate : 30.98%
Test failure rate     : 83.98%

Evil events failure rate   : 94.66%
Benign events failure rate : 28.57%

Return value features done ✅


In [4]:
# Learn rarity from TRAINING data only — never peek at test
process_counts = train_df['processName'].value_counts()
total_train    = len(train_df)
process_freq   = (process_counts / total_train)

def add_process_rarity(df, freq_map):
    df = df.copy()
    # Processes unseen in training get minimum frequency (very rare = suspicious)
    df['process_freq']   = df['processName'].map(freq_map).fillna(0.0)
    df['process_rarity'] = 1 - df['process_freq']   # high rarity = more suspicious
    df['is_rare_process']= (df['process_freq'] < 0.001).astype(int)  # < 0.1% = rare
    return df

train_df = add_process_rarity(train_df, process_freq)
val_df   = add_process_rarity(val_df, process_freq)
test_df  = add_process_rarity(test_df, process_freq)

print("=== TOP 10 RAREST PROCESSES (in test set) ===")
rare_test = test_df[test_df['is_rare_process']==1]['processName'].value_counts().head(10)
print(rare_test)

print(f"\nRare process rate in evil events  : {test_df[test_df['evil']==1]['is_rare_process'].mean()*100:.2f}%")
print(f"Rare process rate in benign events: {test_df[test_df['evil']==0]['is_rare_process'].mean()*100:.2f}%")
print("\nProcess rarity features done ✅")

=== TOP 10 RAREST PROCESSES (in test set) ===
processName
tsm                149155
landscape-sysin      5668
hwe-support-sta      1587
lsb_release          1324
w                    1006
apt-config            855
grep                  732
sh                    573
passwd                544
(systemd)             506
Name: count, dtype: int64

Rare process rate in evil events  : 99.46%
Rare process rate in benign events: 40.72%

Process rarity features done ✅


In [5]:
# Learn normal parent→child pairs from training only
normal_pairs = set(
    zip(train_df['parentProcessId'].astype(str),
        train_df['processName'])
)

def add_parent_child_anomaly(df, known_pairs):
    df = df.copy()
    pairs = list(zip(df['parentProcessId'].astype(str), df['processName']))
    df['unknown_parent_child'] = [0 if p in known_pairs else 1 for p in pairs]
    return df

train_df = add_parent_child_anomaly(train_df, normal_pairs)
val_df   = add_parent_child_anomaly(val_df, normal_pairs)
test_df  = add_parent_child_anomaly(test_df, normal_pairs)

print("=== PARENT-CHILD ANOMALY ===")
print(f"Unknown pairs in training : {train_df['unknown_parent_child'].sum():,}  (should be ~0)")
print(f"Unknown pairs in test     : {test_df['unknown_parent_child'].sum():,}")
print(f"\nUnknown pair rate in evil events  : {test_df[test_df['evil']==1]['unknown_parent_child'].mean()*100:.2f}%")
print(f"Unknown pair rate in benign events: {test_df[test_df['evil']==0]['unknown_parent_child'].mean()*100:.2f}%")
print("\nParent-child anomaly feature done ✅")

=== PARENT-CHILD ANOMALY ===
Unknown pairs in training : 0  (should be ~0)
Unknown pairs in test     : 180,114

Unknown pair rate in evil events  : 100.00%
Unknown pair rate in benign events: 71.01%

Parent-child anomaly feature done ✅


In [6]:
import math

def shannon_entropy(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    freq = {}
    for ch in text:
        freq[ch] = freq.get(ch, 0) + 1
    length = len(text)
    return -sum((c/length) * math.log2(c/length) for c in freq.values())

# Apply to all splits (may take 30-60 seconds)
print("Computing Shannon entropy... (this takes ~30 seconds)")
for df_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df['args_entropy'] = df['args'].apply(shannon_entropy)
    print(f"{df_name} entropy computed ✅")

print("\n=== ENTROPY STATS ===")
print(f"Avg entropy - evil events  : {test_df[test_df['evil']==1]['args_entropy'].mean():.4f}")
print(f"Avg entropy - benign events: {test_df[test_df['evil']==0]['args_entropy'].mean():.4f}")
print("\nShannon entropy feature done ✅")

Computing Shannon entropy... (this takes ~30 seconds)
train entropy computed ✅
val entropy computed ✅
test entropy computed ✅

=== ENTROPY STATS ===
Avg entropy - evil events  : 4.6642
Avg entropy - benign events: 4.5182

Shannon entropy feature done ✅


In [7]:
FEATURE_COLS = [
    'uid_root', 'uid_daemon', 'uid_external',
    'is_failure', 'is_success',
    'process_freq', 'process_rarity', 'is_rare_process',
    'unknown_parent_child',
    'args_entropy'
]

X_train = train_df[FEATURE_COLS].fillna(0)
X_val   = val_df[FEATURE_COLS].fillna(0)
X_test  = test_df[FEATURE_COLS].fillna(0)

# Labels (only test set has meaningful ones)
y_test_sus  = test_df['sus'].values
y_test_evil = test_df['evil'].values

print("=== FINAL FEATURE MATRIX SHAPES ===")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nFeatures: {FEATURE_COLS}")
print(f"\nClass balance in test:")
print(f"  evil=1 : {y_test_evil.sum():,} ({y_test_evil.mean()*100:.2f}%)")
print(f"  evil=0 : {(1-y_test_evil).sum():,} ({(1-y_test_evil).mean()*100:.2f}%)")

=== FINAL FEATURE MATRIX SHAPES ===
X_train : (763144, 10)
X_val   : (188967, 10)
X_test  : (188967, 10)

Features: ['uid_root', 'uid_daemon', 'uid_external', 'is_failure', 'is_success', 'process_freq', 'process_rarity', 'is_rare_process', 'unknown_parent_child', 'args_entropy']

Class balance in test:
  evil=1 : 158,432 (83.84%)
  evil=0 : 30,535 (16.16%)


In [8]:
# Save processed feature matrices
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_val.to_csv(PROCESSED_DIR / "X_val.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)

# Save labels
pd.Series(y_test_sus, name='sus').to_csv(PROCESSED_DIR / "y_test_sus.csv", index=False)
pd.Series(y_test_evil, name='evil').to_csv(PROCESSED_DIR / "y_test_evil.csv", index=False)

# Save process frequency map for inference later
process_freq.to_json(PROCESSED_DIR / "process_freq_map.json")

# Save known parent-child pairs count
with open(PROCESSED_DIR / "feature_metadata.json", 'w') as f:
    json.dump({
        'feature_cols': FEATURE_COLS,
        'n_known_pairs': len(normal_pairs),
        'n_train_rows': len(X_train),
        'rare_threshold': 0.001
    }, f, indent=2)

print("All feature files saved to data/processed/ ✅")
print(f"Files: {[f.name for f in PROCESSED_DIR.glob('*')]}")

All feature files saved to data/processed/ ✅
Files: ['.gitkeep', 'class_imbalance.png', 'dataset_summary.json', 'feature_metadata.json', 'process_freq_map.json', 'X_test.csv', 'X_train.csv', 'X_val.csv', 'y_test_evil.csv', 'y_test_sus.csv']
